In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

ROI = "HIP"
CHEMS = None

from scipy.stats import spearmanr
import pyarrow.parquet as pq

def darken(c, f=0.55):
    return tuple(v * f for v in c[:3])

f = f"{SLIM_DIR}/{ROI}_slim.parquet"
names = pq.ParquetFile(f).schema.names
genes = [g for g in GRMS if g in names]
d = pd.read_parquet(f, columns=["class", "subclass", "chem"] + genes)
d = d[[is_neuron_class(c) for c in d["class"]]]
rname = f"{NAMES.get(ROI, ROI)} ({ROI})"
sub = d.groupby("subclass")[genes].mean()
sub["n_cells"] = d.groupby("subclass").size()
sub["class"] = d.groupby("subclass")["class"].first()
sub = sub[sub["n_cells"] >= MIN_CELLS]
print(rname, len(d), "neurons;", len(sub), "subclasses")
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
rows = []
for ax, (A, B) in zip(axes.ravel(), PAIRS):
    if A not in sub.columns or B not in sub.columns:
        ax.axis("off")
        continue
    qa, qb = sub[A].quantile(0.75), sub[B].quantile(0.75)
    dh = (sub[A] >= qa) & (sub[B] >= qb)
    ax.axvspan(qa, sub[A].max() * 1.05 + 0.1, ymin=0, ymax=1, color="0.92", zorder=0)
    ax.axhspan(qb, sub[B].max() * 1.05 + 0.1, color="0.92", zorder=0)
    ax.scatter(sub[A], sub[B], s=np.clip(sub["n_cells"] / 60, 18, 160), c=[class_color(c) for c in sub["class"]], edgecolor="k", linewidth=0.3, zorder=3)
    rho = spearmanr(sub[A], sub[B])[0]
    cand = sub[dh].assign(score=lambda x: x[A].rank() + x[B].rank()).sort_values("score", ascending=False).head(6)
    for s_, r in cand.iterrows():
        ax.text(r[A], r[B], s_, fontsize=9, color=darken(class_color(r["class"])), zorder=5, ha="left", va="bottom",
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.7))
    ax.set_xlabel(f"{A}: mean log2(CPM+1) per subclass", fontsize=10)
    ax.set_ylabel(f"{B}: mean log2(CPM+1) per subclass", fontsize=10)
    ax.set_title(f"{A} vs {B}: {int(dh.sum())} double-high of {len(sub)} subclasses\nSpearman across subclasses = {rho:.2f}", fontsize=10)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for s_, r in sub.iterrows():
        rows.append((ROI, A, B, s_, r["class"], int(r["n_cells"]), r[A], r[B], bool(dh[s_])))
fig.suptitle(f"{rname}: Grm co-expression across neuronal subclasses (Allen WMB, 10x; subclasses with >= {MIN_CELLS} cells)\ndot = subclass (size = cells, color = class); grey = top quartile of subclass means for that receptor; double-high = in both grey zones", fontsize=11)
fig.subplots_adjust(hspace=0.32, top=0.9)
os.makedirs(f"{RES_DIR}/coexpression", exist_ok=True)
pd.DataFrame(rows, columns=["roi", "gene_a", "gene_b", "subclass", "class", "n_cells", "mean_log2_a", "mean_log2_b", "double_high"]).to_csv(f"{RES_DIR}/coexpression/coexpr_subclass_{ROI}.csv", index=False)
fig.savefig(f"{FIG_DIR}/{ROI}_fig8_coexpr_subclass.png", dpi=170, bbox_inches="tight")
plt.show()